In [3]:
import re
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

# Ensure punkt tokenizer is downloaded
nltk.download('punkt', quiet =True)
nltk.download('punkt_tab', quiet = True)


True

In [5]:
def load_and_clean_parquet(file_path:str = "/content/competitor_app_reviews.parquet")-> pd.DataFrame:
    """
    Loads the Play Store parquet dataset, standardizes columns, and removes noise.
    """
    df = pd.read_parquet(file_path)
    print(f"Loaded raw dataset with shape: {df.shape}")
    print(f"Columns found: {list(df.columns)}")

    #Auto detect and standardize column names from common play store scraper schema
    col_mapping = {}
    for col in df.columns:
      c_lower = col.lower()
      if c_lower in ["content", "review", "text", "review_text", "body"]:
            if "review_text" not in col_mapping.values():
                col_mapping[col] = "review_text"
      elif c_lower in ["app_name", "app", "title", "app_id", "package_name"]:
            if "app_name" not in col_mapping.values():
                col_mapping[col] = "app_name"
      elif c_lower in ["score", "rating", "stars", "star"]:
            if "rating" not in col_mapping.values():
                col_mapping[col] = "rating"
      elif c_lower in ["at", "date", "timestamp", "review_date", "created_at"]:
            if "review_date" not in col_mapping.values():
                col_mapping[col] = "review_date"
      elif c_lower in ["reviewid", "review_id", "id"]:
            if "review_id" not in col_mapping.values():
                col_mapping[col] = "review_id"
    df = df.rename(columns=col_mapping)

    # Normalize App Names if package IDs are used in the column
    if "app_name" in df.columns:
        df["app_name"] = df["app_name"].astype(str).apply(
            lambda x: "Spotify" if "spotify" in x.lower()
            else ("YouTube Music" if "youtube" in x.lower() else x)
        )

    # Drop empty reviews
    df = df.dropna(subset=["review_text"]).copy()
    df["review_text"] = df["review_text"].astype(str).str.strip()

    # Filter out spam, weather copy-pastes, and keyboard smashes
    def is_valid_review(text: str) -> bool:
        if len(text) < 5:
            return False
        if "FIRST BIG COLD FRONT THIS WEEKEND" in text.upper():
            return False
        # Remove single-character repetition spam (e.g., "zzzzz xxxxx")
        alpha_ratio = sum(c.isalpha() for c in text) / max(len(text), 1)
        return alpha_ratio > 0.4

    df = df[df["review_text"].apply(is_valid_review)].reset_index(drop=True)
    print(f"Cleaned dataset shape: {df.shape}")
    return df


def split_reviews_into_clauses(df: pd.DataFrame) -> pd.DataFrame:
    """
    Splits multi-sentence or contrastive reviews ('Great sound, but terrible ads')
    into granular clauses for accurate aspect-level sentiment assignment.
    """
    records = []
    contrastive_pattern = re.compile(r"\b(?:but|however|although|though|except|yet)\b", re.IGNORECASE)

    for idx, row in df.iterrows():
        sentences = sent_tokenize(row["review_text"])
        for sent in sentences:
            # Further split on contrastive conjunctions so positive and negative aspects separate
            sub_clauses = contrastive_pattern.split(sent)
            for clause in sub_clauses:
                clause = clause.strip()
                if len(clause) >= 4:
                    records.append({
                        "review_id": row.get("review_id", idx),
                        "app_name": row.get("app_name", "Unknown"),
                        "rating": row.get("rating", None),
                        "review_date": row.get("review_date", None),
                        "full_review": row["review_text"],
                        "clause": clause
                    })

    clause_df = pd.DataFrame(records)
    print(f"Extracted {len(clause_df)} aspect-ready clauses from {len(df)} reviews.")
    return clause_df


if __name__ == "__main__":
    df_clean = load_and_clean_parquet("competitor_app_reviews.parquet")
    df_clauses = split_reviews_into_clauses(df_clean)
    df_clauses.to_parquet("cleaned_review_clauses.parquet", index=False)

Loaded raw dataset with shape: (10977, 12)
Columns found: ['app_name', 'app_id', 'review_id', 'user_name', 'content', 'score', 'thumbs_up_count', 'review_created_version', 'created_at', 'clean_text', 'word_count', 'date']
Cleaned dataset shape: (10957, 12)
Extracted 23653 aspect-ready clauses from 10957 reviews.


In [6]:
print(df_clean.head(100))

   app_name             app_id                             review_id  \
0   Spotify  com.spotify.music  42b1839a-cc34-4644-a6b4-5de5ff57cd25   
1   Spotify  com.spotify.music  d24b6682-85a9-4a20-b5c0-a4805f7729c0   
2   Spotify  com.spotify.music  b675db44-9b60-4188-be51-2743ee1421e6   
3   Spotify  com.spotify.music  74849c86-9690-44a9-9402-5f1da4e6e6e6   
4   Spotify  com.spotify.music  6098082e-ea85-4601-a9a4-c675523611fb   
..      ...                ...                                   ...   
95  Spotify  com.spotify.music  de2e59b7-0743-4074-9132-1c3418ea8d54   
96  Spotify  com.spotify.music  44e8be4c-fd3f-4742-94f8-9750cdd03b8a   
97  Spotify  com.spotify.music  fec12ce0-0077-4510-99d9-f0afa52a9ef8   
98  Spotify  com.spotify.music  8026173d-a986-4162-b56e-d0fa5d637469   
99  Spotify  com.spotify.music  22e45f88-8935-4617-99c8-9ef25286c5af   

              user_name                                        review_text  \
0            Gavy Lohat                  good but little 

In [7]:
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

# Stakeholder Aspect Taxonomy (Includes English + common Hinglish/Spanish/Indonesian terms in dataset)
ASPECT_TAXONOMY = {
    "Monetization, Ads & Free Tier": [
        r"\bads?\b", r"\badvertis\w*", r"\bpremium\b", r"\bsubscri\w*", r"\bpay\w*",
        r"\bprice\b", r"\bmoney\b", r"\bfree\s*tier\b", r"\bskips?\b", r"\bbackground\s*play\b",
        r"\bgreedy\b", r"\biklan\b", r"\banuncios?\b", r"\breclame\b", r"\bpaisa\b"
    ],
    "Shuffle, Queue & Recommendations": [
        r"\bshuffl\w*", r"\bsmart\s*shuffle\b", r"\bqueue\w*", r"\brecommend\w*",
        r"\balgorithm\w*", r"\brepeat\w*", r"\bloop\w*", r"\bai\s*dj\b", r"\bautoplay\b",
        r"\bmix(?:es)?\b", r"\bdiscover\s*weekly\b", r"\bdaylist\b"
    ],
    "Audio Quality & Playback": [
        r"\baudio\b", r"\bsound\s*quality\b", r"\blossless\b", r"\bdolby\b", r"\batmos\b",
        r"\bequalizer\b", r"\bcrossfade\b", r"\bvolume\b", r"\bbit\s*rate\b", r"\b320\s*kbps\b",
        r"\bflac\b", r"\bbass\b", r"\bgapless\b"
    ],
    "Catalog, Podcasts & Content": [
        r"\bpodcast\w*", r"\baudiobook\w*", r"\bcatalog\w*", r"\blibrary\b", r"\blyrics?\b",
        r"\bremix\w*", r"\bcovers?\b", r"\bexplicit\b", r"\bcensor\w*", r"\bai\s*slop\b",
        r"\bai\s*generat\w*", r"\bsongs?\s*missing\b", r"\bepisode\w*"
    ],
    "UI/UX, Widgets & Organization": [
        r"\bui\b", r"\bux\b", r"\binterface\b", r"\bwidget\w*", r"\b4x1\b", r"\b4x2\b",
        r"\bsort\w*", r"\bfolder\w*", r"\bsearch\b", r"\blayout\b", r"\bdesign\b",
        r"\bsound\s*capsule\b", r"\bplaylist\w*", r"\bnavigat\w*"
    ],
    "App Stability, Offline & Storage": [
        r"\boffline\b", r"\bdownload\w*", r"\bcache\b", r"\bstorage\b", r"\bcrash\w*",
        r"\blag\w*", r"\bbug\w*", r"\bfreez\w*", r"\bglitch\w*", r"\bbattery\b",
        r"\bandroid\s*auto\b", r"\bbuffer\w*", r"\bwifi\b", r"\binternet\b"
    ]
}

# Pre-compile regex patterns for fast matching
COMPILED_TAXONOMY = {
    aspect: re.compile("|".join(patterns), re.IGNORECASE)
    for aspect, patterns in ASPECT_TAXONOMY.items()
}


def detect_aspects(clause: str) -> list[str]:
    """Identifies which stakeholder aspects are mentioned in a clause."""
    matched = [
        aspect for aspect, pattern in COMPILED_TAXONOMY.items()
        if pattern.search(clause)
    ]
    return matched


class DeBERTaABSAEngine:
    """
    Runs true Aspect-Based Sentiment Analysis using a DeBERTa-v3 model
    trained on (Sentence, Aspect) pairs.
    """
    def __init__(self, model_name: str = "yangheng/deberta-v3-base-absa-v1.1", batch_size: int = 32):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing ABSA model on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.batch_size = batch_size
        # Standard label mapping for yangheng/deberta-v3-base-absa-v1.1
        self.id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

    def predict_batch(self, texts: list[str], aspects: list[str]) -> list[dict]:
        results = []
        for i in tqdm(range(0, len(texts), self.batch_size), desc="Running ABSA Inference"):
            batch_texts = texts[i : i + self.batch_size]
            batch_aspects = aspects[i : i + self.batch_size]

            # Format as DeBERTa ABSA pair: (text, aspect)
            encoded = self.tokenizer(
                batch_texts,
                batch_aspects,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**encoded)
                probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

            for prob in probs:
                pred_idx = int(prob.argmax())
                sentiment_label = self.id2label[pred_idx]
                # Net sentiment score ranging from -1.0 (pure negative) to +1.0 (pure positive)
                sentiment_score = float(prob[2] - prob[0])
                results.append({
                    "sentiment": sentiment_label,
                    "confidence": float(prob[pred_idx]),
                    "sentiment_score": round(sentiment_score, 4)
                })
        return results


def run_absa_pipeline(clause_parquet_path: str = "cleaned_review_clauses.parquet") -> pd.DataFrame:
    df_clauses = pd.read_parquet(clause_parquet_path)

    # 1. Detect aspects per clause
    df_clauses["aspects"] = df_clauses["clause"].apply(detect_aspects)

    # Filter to clauses that mention at least one target aspect and explode
    df_exploded = df_clauses[df_clauses["aspects"].map(len) > 0].explode("aspects").reset_index(drop=True)
    df_exploded = df_exploded.rename(columns={"aspects": "aspect_category"})
    print(f"Total (Clause, Aspect) pairs to evaluate: {len(df_exploded)}")

    # 2. Run DeBERTa ABSA Inference
    engine = DeBERTaABSAEngine(batch_size=64)
    predictions = engine.predict_batch(
        texts=df_exploded["clause"].tolist(),
        aspects=df_exploded["aspect_category"].tolist()
    )

    pred_df = pd.DataFrame(predictions)
    final_df = pd.concat([df_exploded, pred_df], axis=1)

    final_df.to_parquet("absa_stakeholder_results.parquet", index=False)
    print("Saved ABSA results to 'absa_stakeholder_results.parquet'")
    return final_df


if __name__ == "__main__":
    run_absa_pipeline()

Total (Clause, Aspect) pairs to evaluate: 10932
Initializing ABSA model on cuda...


config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Running ABSA Inference: 100%|██████████| 171/171 [00:58<00:00,  2.93it/s]

Saved ABSA results to 'absa_stakeholder_results.parquet'


In [8]:
import pandas as pd


def generate_stakeholder_summary(results_path: str = "absa_stakeholder_results.parquet") -> pd.DataFrame:
    df = pd.read_parquet(results_path)

    # Calculate counts and percentages per App and Aspect
    summary = (
        df.groupby(["app_name", "aspect_category"])
        .agg(
            total_mentions=("sentiment", "count"),
            positive_count=("sentiment", lambda s: (s == "Positive").sum()),
            neutral_count=("sentiment", lambda s: (s == "Neutral").sum()),
            negative_count=("sentiment", lambda s: (s == "Negative").sum()),
            avg_sentiment_score=("sentiment_score", "mean")
        )
        .reset_index()
    )

    summary["pct_positive"] = (summary["positive_count"] / summary["total_mentions"] * 100).round(1)
    summary["pct_neutral"] = (summary["neutral_count"] / summary["total_mentions"] * 100).round(1)
    summary["pct_negative"] = (summary["negative_count"] / summary["total_mentions"] * 100).round(1)

    # Net Sentiment Score (NSS) = % Positive - % Negative (scales from -100 to +100)
    summary["net_sentiment_score"] = (summary["pct_positive"] - summary["pct_negative"]).round(1)

    summary.to_csv("stakeholder_aspect_comparison.csv", index=False)
    print("\n=== STAKEHOLDER ASPECT COMPARISON SUMMARY ===")
    print(summary[["app_name", "aspect_category", "total_mentions", "pct_positive", "pct_negative", "net_sentiment_score"]])
    return summary


if __name__ == "__main__":
    generate_stakeholder_summary()


=== STAKEHOLDER ASPECT COMPARISON SUMMARY ===
         app_name                   aspect_category  total_mentions  \
0         Spotify  App Stability, Offline & Storage             659   
1         Spotify          Audio Quality & Playback             176   
2         Spotify       Catalog, Podcasts & Content             494   
3         Spotify     Monetization, Ads & Free Tier            2639   
4         Spotify  Shuffle, Queue & Recommendations             636   
5         Spotify     UI/UX, Widgets & Organization             829   
6   YouTube Music  App Stability, Offline & Storage             999   
7   YouTube Music          Audio Quality & Playback             200   
8   YouTube Music       Catalog, Podcasts & Content             566   
9   YouTube Music     Monetization, Ads & Free Tier            1782   
10  YouTube Music  Shuffle, Queue & Recommendations             833   
11  YouTube Music     UI/UX, Widgets & Organization            1119   

    pct_positive  pct_negativ